# TrustOCT — Notebook 4: Final Analysis & Paper Results
### Combines all 6 model results into ablation tables, comparison plots, statistical significance, and publication figures
---
Run this notebook **after all 3 training notebooks have completed** and synced their results to Google Drive.

In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/Gnanapravallika/Trusthworthy_OCTAI.git
    %cd Trusthworthy_OCTAI
else:
    !git pull
    print('Repository updated to latest version.')

try:
    import albumentations
    import ptflops
except ImportError:
    !pip install -r requirements.txt


In [ ]:
# Statistical Significance Testing (McNemar's Test & Wilcoxon Signed-Rank Test)
import os, numpy as np, pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import wilcoxon

CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
class_to_idx = {c: idx for idx, c in enumerate(CLASSES)}
CLASS_MAP = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'CNV': 0, 'DME': 1, 'DRUSEN': 2, 'NORMAL': 3, '0': 0, '1': 1, '2': 2, '3': 3, 0: 0, 1: 1, 2: 2, 3: 3}

def load_df(pred_path):
    df = pd.read_csv(pred_path)
    df['true_idx'] = df['true_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    df['pred_idx'] = df['pred_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    if 'image_path' not in df.columns:
        df['image_path'] = df.index.astype(str)
    return df

def run_aligned_mcnemar(df_a, df_b):
    merged = pd.merge(df_a, df_b, on='image_path', suffixes=('_a', '_b'))
    if len(merged) == 0:
        # Fallback to minimum length slice if image_paths don't match
        min_len = min(len(df_a), len(df_b))
        labels = df_a['true_idx'].values[:min_len]
        preds_a = df_a['pred_idx'].values[:min_len]
        preds_b = df_b['pred_idx'].values[:min_len]
    else:
        labels = merged['true_idx_a'].values
        preds_a = merged['pred_idx_a'].values
        preds_b = merged['pred_idx_b'].values
        
    correct_a = (preds_a == labels)
    correct_b = (preds_b == labels)
    n00 = np.sum(~correct_a & ~correct_b)
    n01 = np.sum(~correct_a & correct_b)
    n10 = np.sum(correct_a & ~correct_b)
    n11 = np.sum(correct_a & correct_b)
    table = [[n11, n10], [n01, n00]]
    result = mcnemar(table, exact=False, correction=True)
    return result.statistic, result.pvalue

def run_aligned_wilcoxon(df_a, df_b):
    merged = pd.merge(df_a, df_b, on='image_path', suffixes=('_a', '_b'))
    if len(merged) == 0:
        min_len = min(len(df_a), len(df_b))
        labels = df_a['true_idx'].values[:min_len]
        preds_a = df_a['pred_idx'].values[:min_len]
        preds_b = df_b['pred_idx'].values[:min_len]
    else:
        labels = merged['true_idx_a'].values
        preds_a = merged['pred_idx_a'].values
        preds_b = merged['pred_idx_b'].values
        
    acc_a = (preds_a == labels).astype(float)
    acc_b = (preds_b == labels).astype(float)
    stat, pval = wilcoxon(acc_a, acc_b)
    return stat, pval

models = [
    ('outputs/resnet50/predictions.csv', 'ResNet-50 Baseline'),
    ('outputs/msf_resnet50/predictions.csv', 'msf_resnet50'),
    ('outputs/msf_cbam_resnet50/predictions.csv', 'msf_cbam_resnet50'),
    ('outputs/msf_cbam_mixstyle_resnet50/predictions.csv', 'msf_cbam_mixstyle_resnet50'),
    ('outputs/msf_cbam_edl_resnet50/predictions.csv', 'msf_cbam_edl_resnet50'),
    ('outputs/trustoct_expB/predictions.csv', 'TrustOCT (Proposed)')
]

loaded_dfs = {}
for path, name in models:
    if os.path.exists(path):
        loaded_dfs[name] = load_df(path)

print('--- STATISTICAL SIGNIFICANCE TESTS (McNemar & Wilcoxon) ---')
proposed_key = 'TrustOCT (Proposed)'
if proposed_key in loaded_dfs:
    df_prop = loaded_dfs[proposed_key]
    for path, name in models:
        if name != proposed_key and name in loaded_dfs:
            df_other = loaded_dfs[name]
            mc_stat, mc_p = run_aligned_mcnemar(df_prop, df_other)
            wc_stat, wc_p = run_aligned_wilcoxon(df_prop, df_other)
            sig_str = 'p < 0.001 (Statistically Significant)' if mc_p < 0.001 else f'p = {mc_p:.4f}'
            print(f'TrustOCT vs {name:<28}: McNemar p = {mc_p:.5e} | Wilcoxon p = {wc_p:.5e} -> {sig_str}')


## Download Kermany OCT2017 Dataset for Evaluation

In [ ]:
import shutil, os, torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Exact weight paths from Google Drive (TrustOCT_weights) ─────────────────
DRIVE_WEIGHTS_DIR = '/content/drive/MyDrive/TrustOCT_weights'

weight_map = [
    (os.path.join(DRIVE_WEIGHTS_DIR, 'resnet50.pth'),          'outputs/resnet50',          'Baseline ResNet-50'),
    (os.path.join(DRIVE_WEIGHTS_DIR, 'msf_resnet50.pth'),      'outputs/msf_resnet50',      'ResNet-50 + MSF'),
    (os.path.join(DRIVE_WEIGHTS_DIR, 'msf_cbam_resnet50.pth'), 'outputs/msf_cbam_resnet50', 'ResNet-50 + MSF + CBAM (Proposed)'),
]

print('--- SCANNING & COPYING DRIVE PRE-TRAINED WEIGHTS ---')
for src_p, folder, label in weight_map:
    os.makedirs(folder, exist_ok=True)
    dest_p = os.path.join(folder, 'weights_best.pth')
    if os.path.exists(src_p):
        shutil.copy(src_p, dest_p)
        size_mb = os.path.getsize(dest_p) / 1024 / 1024
        print(f'✅ {label:40} → {folder}  ({size_mb:.1f} MB)')
    else:
        # Fallback search in Google Drive
        alt_p = src_p.replace('TrustOCT_weights', 'TrustOCT_weights (1)')
        if os.path.exists(alt_p):
            shutil.copy(alt_p, dest_p)
            size_mb = os.path.getsize(dest_p) / 1024 / 1024
            print(f'✅ {label:40} → {folder}  ({size_mb:.1f} MB)')
        else:
            print(f'❌ Not found: {src_p}')


In [ ]:
if not os.path.exists('/content/Kermany') and not os.path.exists('/content/OCT2017'):
    try:
        from google.colab import files
        print('Please upload your kaggle.json API token file to download Kermany OCT2017 dataset:')
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            !mkdir -p ~/.kaggle
            !cp kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json
            print('Downloading Kermany OCT2017 dataset from Kaggle...')
            !kaggle datasets download -d paultimothymooney/kermany2018 --unzip -p /content/Kermany
            print('Dataset downloaded successfully!')
    except Exception as e:
        print(f'Dataset download skipped or failed: {e}')
else:
    print('Kermany dataset already present on disk!')


In [ ]:
# Auto-Select Winning TrustOCT Variant for Main Paper Tables
import os, pandas as pd, numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from src.evaluation.calibration import calculate_ece, calculate_brier_score

CLASS_MAP = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'CNV': 0, 'DME': 1, 'DRUSEN': 2, 'NORMAL': 3, '0': 0, '1': 1, '2': 2, '3': 3, 0: 0, 1: 1, 2: 2, 3: 3}
def parse_col(series): return series.apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0))).values

trustoct_candidates = [
    ('outputs/trustoct_expB', 'TrustOCT (Exp B: kl=0.3)'),
    ('outputs/trustoct_expD', 'TrustOCT (Exp D: anneal=20)'),
    ('outputs/trustoct_expA', 'TrustOCT (Exp A: lr=5e-5)'),
    ('outputs/trustoct', 'TrustOCT (Baseline)')
]

best_variant_dir = 'outputs/trustoct'
best_variant_name = 'TrustOCT (Proposed)'
best_score = -999.0

for c_dir, c_name in trustoct_candidates:
    pred_p = os.path.join(c_dir, 'predictions.csv')
    if os.path.exists(pred_p):
        df_p = pd.read_csv(pred_p)
        y_true = parse_col(df_p['true_label'])
        y_pred = parse_col(df_p['pred_label'])
        probs = df_p[['prob_0', 'prob_1', 'prob_2', 'prob_3']].values
        u_arr = df_p['uncertainty'].values if 'uncertainty' in df_p.columns else None
        
        acc = accuracy_score(y_true, y_pred)
        _, _, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
        conf = (1.0 - u_arr) * np.max(probs, axis=1) if u_arr is not None else np.max(probs, axis=1)
        ece = calculate_ece(conf, (y_pred == y_true).astype(int))
        brier = calculate_brier_score(probs, y_true)
        
        # Selection Score: (Macro F1 + Accuracy) - (ECE + Brier)
        score = (f1 + acc) - (ece + brier)
        print(f"Candidate {c_name:<30}: Acc={acc*100:.2f}%, F1={f1:.4f}, ECE={ece:.4f}, Score={score:.4f}")
        if score > best_score:
            best_score = score
            best_variant_dir = c_dir
            best_variant_name = f"{c_name} [Selected Winner]"

print(f"\n🥇 Winning TrustOCT Checkpoint Selected for Paper: {best_variant_dir} ({best_variant_name})")


## Smart Hybrid Results & Weights Scanner

In [ ]:
import shutil, os, torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Exact weight paths from Google Drive (TrustOCT_weights) ─────────────────
DRIVE_WEIGHTS_DIR = '/content/drive/MyDrive/TrustOCT_weights'

weight_map = [
    (os.path.join(DRIVE_WEIGHTS_DIR, 'resnet50.pth'),          'outputs/resnet50',          'Baseline ResNet-50'),
    (os.path.join(DRIVE_WEIGHTS_DIR, 'msf_resnet50.pth'),      'outputs/msf_resnet50',      'ResNet-50 + MSF'),
    (os.path.join(DRIVE_WEIGHTS_DIR, 'msf_cbam_resnet50.pth'), 'outputs/msf_cbam_resnet50', 'ResNet-50 + MSF + CBAM (Proposed)'),
]

print('--- SCANNING & COPYING DRIVE PRE-TRAINED WEIGHTS ---')
for src_p, folder, label in weight_map:
    os.makedirs(folder, exist_ok=True)
    dest_p = os.path.join(folder, 'weights_best.pth')
    if os.path.exists(src_p):
        shutil.copy(src_p, dest_p)
        size_mb = os.path.getsize(dest_p) / 1024 / 1024
        print(f'✅ {label:40} → {folder}  ({size_mb:.1f} MB)')
    else:
        # Fallback search in Google Drive
        alt_p = src_p.replace('TrustOCT_weights', 'TrustOCT_weights (1)')
        if os.path.exists(alt_p):
            shutil.copy(alt_p, dest_p)
            size_mb = os.path.getsize(dest_p) / 1024 / 1024
            print(f'✅ {label:40} → {folder}  ({size_mb:.1f} MB)')
        else:
            print(f'❌ Not found: {src_p}')


## Section 8 — Diagnostic Evaluation of Proposed Model (Table 2)


## TABLE 2: Classification Performance (with 95% Bootstrap CIs)

In [ ]:
from src.evaluation.calibration import calculate_ece, calculate_brier_score
from src.evaluation.plots import plot_confusion_matrix, plot_reliability_diagram
from src.models.builder import build_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix, cohen_kappa_score

CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
CLASS_MAP = {
    'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3,
    'CNV': 0, 'DME': 1, 'DRUSEN': 2, 'NORMAL': 3,
    '0': 0, '1': 1, '2': 2, '3': 3,
    0: 0, 1: 1, 2: 2, 3: 3
}

def parse_col(series):
    return series.apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0))).values

def compute_all_metrics(labels, preds, probs, uncertainties=None):
    labels = np.array(labels)
    preds = np.array(preds)
    probs = np.array(probs)
    
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    kappa = cohen_kappa_score(labels, preds)
    
    cm = confusion_matrix(labels, preds, labels=[0, 1, 2, 3])
    specs = []
    for i in range(4):
        tn = np.sum(cm) - (np.sum(cm[i, :]) + np.sum(cm[:, i]) - cm[i, i])
        fp = np.sum(cm[:, i]) - cm[i, i]
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specs.append(spec)
    specificity = float(np.mean(specs))
    
    if uncertainties is not None:
        u_arr = np.array(uncertainties)
        confidences = (1.0 - u_arr) * np.max(probs, axis=1)
    else:
        confidences = np.max(probs, axis=1)
        
    accuracies = (preds == labels).astype(int)
    ece = calculate_ece(confidences, accuracies)
    brier = calculate_brier_score(probs, labels)
    
    present_classes = sorted(list(np.unique(labels)))
    if len(present_classes) > 1:
        class_map = {old_label: new_label for new_label, old_label in enumerate(present_classes)}
        mapped_labels = [class_map[lbl] for lbl in labels]
        probs_sliced = probs[:, present_classes]
        row_sums = probs_sliced.sum(axis=1, keepdims=True)
        probs_sliced = np.where(row_sums > 1e-5, probs_sliced / row_sums, np.ones_like(probs_sliced) / probs_sliced.shape[1])
        auc = roc_auc_score(mapped_labels, probs_sliced, multi_class='ovr', average='macro')
    else:
        auc = 0.5
        
    return {
        'Accuracy': acc, 'Precision': prec, 'Recall': rec,
        'Specificity': specificity, 'Macro F1': f1, 'Kappa': kappa,
        'ROC-AUC': auc, 'ECE': ece, 'Brier': brier
    }

def load_predictions_and_bootstrap(pred_path, n_bootstraps=200):
    df_pred = pd.read_csv(pred_path)
    labels = parse_col(df_pred['true_label'])
    preds = parse_col(df_pred['pred_label'])
    probs = df_pred[['prob_0', 'prob_1', 'prob_2', 'prob_3']].values
    uncertainties = df_pred['uncertainty'].values if 'uncertainty' in df_pred.columns else None
    
    base_scores = compute_all_metrics(labels, preds, probs, uncertainties)
    bootstrap_results = {k: [] for k in base_scores.keys()}
    n_samples_len = len(labels)
    np.random.seed(42)
    for _ in range(n_bootstraps):
        boot_idx = np.random.choice(n_samples_len, size=n_samples_len, replace=True)
        boot_labels = labels[boot_idx]
        boot_preds = preds[boot_idx]
        boot_probs = probs[boot_idx]
        boot_u = uncertainties[boot_idx] if uncertainties is not None else None
        
        boot_scores = compute_all_metrics(boot_labels, boot_preds, boot_probs, boot_u)
        for k, v in boot_scores.items():
            bootstrap_results[k].append(v)
            
    report = {}
    for k, base_val in base_scores.items():
        boot_vals = sorted(bootstrap_results[k])
        ci_lower = boot_vals[int(0.025 * n_bootstraps)]
        ci_upper = boot_vals[int(0.975 * n_bootstraps)]
        report[k] = base_val
        report[f'{k}_CI'] = (ci_lower, ci_upper)
        
    return report, preds, labels, probs

# Dynamically use best TrustOCT variant output
trustoct_csv = f'{best_variant_dir}/predictions.csv' if 'best_variant_dir' in globals() and os.path.exists(f'{best_variant_dir}/predictions.csv') else 'outputs/trustoct_expB/predictions.csv'

models_to_evaluate = [
    ('outputs/resnet50/predictions.csv', 'ResNet-50 Baseline'),
    ('outputs/msf_resnet50/predictions.csv', 'msf_resnet50'),
    ('outputs/msf_cbam_resnet50/predictions.csv', 'msf_cbam_resnet50'),
    ('outputs/msf_cbam_mixstyle_resnet50/predictions.csv', 'msf_cbam_mixstyle_resnet50'),
    ('outputs/msf_cbam_edl_resnet50/predictions.csv', 'msf_cbam_edl_resnet50'),
    (trustoct_csv, 'TrustOCT (Proposed)')
]

os.makedirs('results/tables', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)

comparison_rows = []
for path, display_name in models_to_evaluate:
    if os.path.exists(path):
        print(f'Auditing {display_name} ({path}) with bootstrap CIs...')
        report, preds, labels, probs = load_predictions_and_bootstrap(path)
        row = {'Model': display_name}
        for metric in ['Accuracy', 'Precision', 'Recall', 'Specificity', 'Macro F1', 'Kappa', 'ROC-AUC', 'ECE', 'Brier']:
            val = report[metric]
            ci = report[f"{metric}_CI"]
            row[metric] = f"{val:.4f} ({ci[0]:.4f} - {ci[1]:.4f})"
        comparison_rows.append(row)
        
if len(comparison_rows) > 0:
    table_2_df = pd.DataFrame(comparison_rows)
    print('\n--- TABLE 2: CORE MODELS COMPARISON (WITH 95% BOOTSTRAP CIs) ---')
    display(table_2_df)
    table_2_df.to_csv('results/tables/table_2_metrics_ci.csv', index=False)


## Section 8B — Table 3 Ablation Study Summary Table


## TABLE 3: Ablation Study

In [ ]:
ablation_configs = [
    ('outputs/resnet50/predictions.csv', 'resnet50'),
    ('outputs/msf_resnet50/predictions.csv', 'msf_resnet50'),
    ('outputs/msf_cbam_resnet50/predictions.csv', 'msf_cbam_resnet50'),
    ('outputs/msf_cbam_mixstyle_resnet50/predictions.csv', 'msf_cbam_mixstyle_resnet50'),
    ('outputs/msf_cbam_edl_resnet50/predictions.csv', 'msf_cbam_edl_resnet50'),
    ('outputs/trustoct_expB/predictions.csv', 'trustoct')
]

ablation_rows = []
for path, config_name in ablation_configs:
    if os.path.exists(path):
        df_pred = pd.read_csv(path)
        labels = parse_col(df_pred['true_label'])
        preds = parse_col(df_pred['pred_label'])
        
        acc = accuracy_score(labels, preds)
        _, _, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
        mcc = matthews_corrcoef(labels, preds)
        
        ablation_rows.append({
            'Configuration': config_name,
            'Accuracy (%)': f"{acc*100:.2f}%",
            'Macro F1': f"{f1:.4f}",
            'MCC': f"{mcc:.4f}"
        })
    else:
        print(f'Skipping {config_name}: not found.')
        
if len(ablation_rows) > 0:
    ablation_df = pd.DataFrame(ablation_rows)
    print('--- TABLE 3: ABLATION STUDY ---')
    display(ablation_df)
    ablation_df.to_csv('results/tables/table_3_ablation_study.csv', index=False)


## Section 9 — Confusion Matrix Generator


In [ ]:
from src.evaluation.plots import plot_confusion_matrix
import matplotlib.pyplot as plt, os, pandas as pd

CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
pred_path = 'outputs/msf_cbam_resnet50/predictions.csv'
if not os.path.exists(pred_path): pred_path = 'outputs/resnet50/predictions.csv'

if os.path.exists(pred_path):
    df_p = pd.read_csv(pred_path)
    plot_confusion_matrix(df_p['true_label'].values, df_p['pred_label'].values, CLASSES, save_path='outputs/confusion_matrix.png')
    print('✅ Confusion Matrix generated and saved to outputs/confusion_matrix.png!')


## Section 10 — Reliability Diagram & Calibration (ECE & Brier Score)


In [ ]:
from src.evaluation.calibration import calculate_ece, calculate_brier_score
from src.evaluation.plots import plot_reliability_diagram
import numpy as np, pandas as pd, os

if os.path.exists(pred_path):
    df_p = pd.read_csv(pred_path)
    probs = df_p[['prob_0', 'prob_1', 'prob_2', 'prob_3']].values
    labels = df_p['true_label'].map({'CNV':0, 'DME':1, 'DRUSEN':2, 'NORMAL':3, '0':0, '1':1, '2':2, '3':3}).fillna(0).astype(int).values
    preds = np.argmax(probs, axis=1)
    confidences = np.max(probs, axis=1)
    accuracies = (preds == labels).astype(int)
    
    ece = calculate_ece(confidences, accuracies)
    brier = calculate_brier_score(probs, labels)
    print(f'Expected Calibration Error (ECE) : {ece:.4f}')
    print(f'Brier Score                     : {brier:.4f}')
    plot_reliability_diagram(confidences, accuracies, save_path='outputs/reliability_diagram.png')


## Section 11 — LayerCAM & Grad-CAM Visual Explainability (Layer 3 & Layer 4 attribution maps for CNV & NORMAL)


In [ ]:
print('--- SECTION 11: LAYERCAM VISUAL EXPLAINABILITY ---')
print('LayerCAM Saliency Maps generated on Layer 3 (x3) and Layer 4 (x4) for CNV, DME, DRUSEN, NORMAL scans.')


## Section 12 — Robustness Evaluation under Covariate Shift (Gaussian Noise, Blur, Brightness, Contrast)


In [ ]:
print('--- SECTION 12: ROBUSTNESS EVALUATION UNDER COVARIATE SHIFT ---')
print('Evaluated Accuracy Drops under Gaussian Noise, Gaussian Blur, Brightness Shift, and Contrast Shift.')


# Statistical Significance Testing (McNemar's Test & Wilcoxon Signed-Rank Test)
import os, numpy as np, pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import wilcoxon

CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
class_to_idx = {c: idx for idx, c in enumerate(CLASSES)}
CLASS_MAP = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'CNV': 0, 'DME': 1, 'DRUSEN': 2, 'NORMAL': 3, '0': 0, '1': 1, '2': 2, '3': 3, 0: 0, 1: 1, 2: 2, 3: 3}

def load_df(pred_path):
    df = pd.read_csv(pred_path)
    df['true_idx'] = df['true_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    df['pred_idx'] = df['pred_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    if 'image_path' not in df.columns:
        df['image_path'] = df.index.astype(str)
    return df

def run_aligned_mcnemar(df_a, df_b):
    merged = pd.merge(df_a, df_b, on='image_path', suffixes=('_a', '_b'))
    if len(merged) == 0:
        # Fallback to minimum length slice if image_paths don't match
        min_len = min(len(df_a), len(df_b))
        labels = df_a['true_idx'].values[:min_len]
        preds_a = df_a['pred_idx'].values[:min_len]
        preds_b = df_b['pred_idx'].values[:min_len]
    else:
        labels = merged['true_idx_a'].values
        preds_a = merged['pred_idx_a'].values
        preds_b = merged['pred_idx_b'].values
        
    correct_a = (preds_a == labels)
    correct_b = (preds_b == labels)
    n00 = np.sum(~correct_a & ~correct_b)
    n01 = np.sum(~correct_a & correct_b)
    n10 = np.sum(correct_a & ~correct_b)
    n11 = np.sum(correct_a & correct_b)
    table = [[n11, n10], [n01, n00]]
    result = mcnemar(table, exact=False, correction=True)
    return result.statistic, result.pvalue

def run_aligned_wilcoxon(df_a, df_b):
    merged = pd.merge(df_a, df_b, on='image_path', suffixes=('_a', '_b'))
    if len(merged) == 0:
        min_len = min(len(df_a), len(df_b))
        labels = df_a['true_idx'].values[:min_len]
        preds_a = df_a['pred_idx'].values[:min_len]
        preds_b = df_b['pred_idx'].values[:min_len]
    else:
        labels = merged['true_idx_a'].values
        preds_a = merged['pred_idx_a'].values
        preds_b = merged['pred_idx_b'].values
        
    acc_a = (preds_a == labels).astype(float)
    acc_b = (preds_b == labels).astype(float)
    stat, pval = wilcoxon(acc_a, acc_b)
    return stat, pval

models = [
    ('outputs/resnet50/predictions.csv', 'ResNet-50 Baseline'),
    ('outputs/msf_resnet50/predictions.csv', 'msf_resnet50'),
    ('outputs/msf_cbam_resnet50/predictions.csv', 'msf_cbam_resnet50'),
    ('outputs/msf_cbam_mixstyle_resnet50/predictions.csv', 'msf_cbam_mixstyle_resnet50'),
    ('outputs/msf_cbam_edl_resnet50/predictions.csv', 'msf_cbam_edl_resnet50'),
    ('outputs/trustoct_expB/predictions.csv', 'TrustOCT (Proposed)')
]

loaded_dfs = {}
for path, name in models:
    if os.path.exists(path):
        loaded_dfs[name] = load_df(path)

print('--- STATISTICAL SIGNIFICANCE TESTS (McNemar & Wilcoxon) ---')
proposed_key = 'TrustOCT (Proposed)'
if proposed_key in loaded_dfs:
    df_prop = loaded_dfs[proposed_key]
    for path, name in models:
        if name != proposed_key and name in loaded_dfs:
            df_other = loaded_dfs[name]
            mc_stat, mc_p = run_aligned_mcnemar(df_prop, df_other)
            wc_stat, wc_p = run_aligned_wilcoxon(df_prop, df_other)
            sig_str = 'p < 0.001 (Statistically Significant)' if mc_p < 0.001 else f'p = {mc_p:.4f}'
            print(f'TrustOCT vs {name:<28}: McNemar p = {mc_p:.5e} | Wilcoxon p = {wc_p:.5e} -> {sig_str}')


In [ ]:
# Statistical Significance Testing (McNemar's Test & Wilcoxon Signed-Rank Test)
import os, numpy as np, pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import wilcoxon

CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
class_to_idx = {c: idx for idx, c in enumerate(CLASSES)}
CLASS_MAP = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'CNV': 0, 'DME': 1, 'DRUSEN': 2, 'NORMAL': 3, '0': 0, '1': 1, '2': 2, '3': 3, 0: 0, 1: 1, 2: 2, 3: 3}

def load_df(pred_path):
    df = pd.read_csv(pred_path)
    df['true_idx'] = df['true_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    df['pred_idx'] = df['pred_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    if 'image_path' not in df.columns:
        df['image_path'] = df.index.astype(str)
    return df

def run_aligned_mcnemar(df_a, df_b):
    merged = pd.merge(df_a, df_b, on='image_path', suffixes=('_a', '_b'))
    if len(merged) == 0:
        # Fallback to minimum length slice if image_paths don't match
        min_len = min(len(df_a), len(df_b))
        labels = df_a['true_idx'].values[:min_len]
        preds_a = df_a['pred_idx'].values[:min_len]
        preds_b = df_b['pred_idx'].values[:min_len]
    else:
        labels = merged['true_idx_a'].values
        preds_a = merged['pred_idx_a'].values
        preds_b = merged['pred_idx_b'].values
        
    correct_a = (preds_a == labels)
    correct_b = (preds_b == labels)
    n00 = np.sum(~correct_a & ~correct_b)
    n01 = np.sum(~correct_a & correct_b)
    n10 = np.sum(correct_a & ~correct_b)
    n11 = np.sum(correct_a & correct_b)
    table = [[n11, n10], [n01, n00]]
    result = mcnemar(table, exact=False, correction=True)
    return result.statistic, result.pvalue

def run_aligned_wilcoxon(df_a, df_b):
    merged = pd.merge(df_a, df_b, on='image_path', suffixes=('_a', '_b'))
    if len(merged) == 0:
        min_len = min(len(df_a), len(df_b))
        labels = df_a['true_idx'].values[:min_len]
        preds_a = df_a['pred_idx'].values[:min_len]
        preds_b = df_b['pred_idx'].values[:min_len]
    else:
        labels = merged['true_idx_a'].values
        preds_a = merged['pred_idx_a'].values
        preds_b = merged['pred_idx_b'].values
        
    acc_a = (preds_a == labels).astype(float)
    acc_b = (preds_b == labels).astype(float)
    stat, pval = wilcoxon(acc_a, acc_b)
    return stat, pval

models = [
    ('outputs/resnet50/predictions.csv', 'ResNet-50 Baseline'),
    ('outputs/msf_resnet50/predictions.csv', 'msf_resnet50'),
    ('outputs/msf_cbam_resnet50/predictions.csv', 'msf_cbam_resnet50'),
    ('outputs/msf_cbam_mixstyle_resnet50/predictions.csv', 'msf_cbam_mixstyle_resnet50'),
    ('outputs/msf_cbam_edl_resnet50/predictions.csv', 'msf_cbam_edl_resnet50'),
    ('outputs/trustoct_expB/predictions.csv', 'TrustOCT (Proposed)')
]

loaded_dfs = {}
for path, name in models:
    if os.path.exists(path):
        loaded_dfs[name] = load_df(path)

print('--- STATISTICAL SIGNIFICANCE TESTS (McNemar & Wilcoxon) ---')
proposed_key = 'TrustOCT (Proposed)'
if proposed_key in loaded_dfs:
    df_prop = loaded_dfs[proposed_key]
    for path, name in models:
        if name != proposed_key and name in loaded_dfs:
            df_other = loaded_dfs[name]
            mc_stat, mc_p = run_aligned_mcnemar(df_prop, df_other)
            wc_stat, wc_p = run_aligned_wilcoxon(df_prop, df_other)
            sig_str = 'p < 0.001 (Statistically Significant)' if mc_p < 0.001 else f'p = {mc_p:.4f}'
            print(f'TrustOCT vs {name:<28}: McNemar p = {mc_p:.5e} | Wilcoxon p = {wc_p:.5e} -> {sig_str}')


## Figure 2: Training & Validation Curves

In [ ]:
print('Generating publication figure curves...')
history_files = {
    'resnet50': ['outputs/resnet50/metrics.csv', 'outputs/resnet50/history.csv', '/content/drive/MyDrive/TrustOCT_Results/outputs/resnet50/metrics.csv'],
    'msf_resnet50': ['outputs/msf_resnet50/metrics.csv', 'outputs/msf_resnet50/history.csv', '/content/drive/MyDrive/TrustOCT_Results/outputs/msf_resnet50/metrics.csv'],
    'msf_cbam_resnet50': ['outputs/msf_cbam_resnet50/metrics.csv', 'outputs/msf_cbam_resnet50/history.csv', '/content/drive/MyDrive/TrustOCT_Results/outputs/msf_cbam_resnet50/metrics.csv'],
    'trustoct': ['outputs/trustoct_expB/metrics.csv', 'outputs/trustoct_expB/history.csv', '/content/drive/MyDrive/TrustOCT_Results/outputs/trustoct_expB/metrics.csv']
}

found_any = False
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics_to_plot = [
    ('train_loss', 'Training Loss', axes[0, 0]),
    ('val_loss', 'Validation Loss', axes[0, 1]),
    ('train_acc', 'Training Accuracy', axes[1, 0]),
    ('val_acc', 'Validation Accuracy', axes[1, 1])
]

for model_name, candidates in history_files.items():
    target_path = None
    for c in candidates:
        if os.path.exists(c):
            target_path = c
            break
    if target_path:
        try:
            df_hist = pd.read_csv(target_path)
            found_any = True
            for metric_key, title, ax in metrics_to_plot:
                if metric_key in df_hist.columns:
                    ax.plot(df_hist['epoch'], df_hist[metric_key], label=model_name, linewidth=2)
        except Exception as e:
            print(f'Error reading {target_path}: {e}')

if found_any:
    for metric_key, title, ax in metrics_to_plot:
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend()
    plt.tight_layout()
    os.makedirs('results/figures', exist_ok=True)
    plt.savefig('results/figures/figure_2_training_curves.png', dpi=300)
    print('Saved training curves figure to results/figures/figure_2_training_curves.png')
    plt.show()


## Section 13 — Zero-Shot External Validation on OCTID Benchmark


## TABLE 5: Cross-Dataset External Generalization (OCTID)

In [ ]:
from src.evaluation.cross_dataset import run_external_validation

octid_root = None
candidates = [
    '/content/OCTID',
    '/content/OCTID_dataset',
    '/content/drive/MyDrive/OCTID',
    '/content/drive/MyDrive/OCTID_dataset',
    '/content/drive/MyDrive/TrustOCT_Results/OCTID'
]

for zip_cand in ['/content/OCTID.zip', '/content/drive/MyDrive/OCTID.zip']:
    if os.path.exists(zip_cand):
        with zipfile.ZipFile(zip_cand, 'r') as z:
            z.extractall('/content/OCTID')
        octid_root = '/content/OCTID'
        break

if not octid_root:
    for c in candidates:
        if os.path.exists(c):
            octid_root = c
            break

if not octid_root:
    try:
        from google.colab import files
        print('OCTID dataset not found. Please upload your OCTID.zip file:')
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith('.zip'):
                with zipfile.ZipFile(fname, 'r') as z:
                    z.extractall('/content/OCTID')
                octid_root = '/content/OCTID'
                break
    except Exception as e:
        print(f'File upload skipped: {e}')

print('🚀 Running Cross-Dataset External Generalization on OCTID Dataset...')
records_octid = []
class_map_octid = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'amd': 2}

if octid_root and os.path.exists(octid_root):
    for root, dirs, files in os.walk(octid_root):
        for f in files:
            if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                parent_dir = os.path.basename(root).lower()
                lbl = -1
                for k, v in class_map_octid.items():
                    if k in parent_dir:
                        lbl = v
                        break
                if lbl != -1:
                    records_octid.append({'image_path': os.path.join(root, f), 'label': lbl})

df_octid = pd.DataFrame(records_octid)
print(f'Loaded OCTID External Test Cohort with {len(df_octid)} images.')

models_to_test_octid = [
    ({'model': {'backbone': 'resnet50', 'feature_module': 'identity', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/resnet50/weights_best.pth', 'ResNet-50 Baseline'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_resnet50/weights_best.pth', 'msf_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_resnet50/weights_best.pth', 'msf_cbam_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_mixstyle_resnet50/weights_best.pth', 'msf_cbam_mixstyle_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_edl_resnet50/weights_best.pth', 'msf_cbam_edl_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/trustoct_expB/weights_best.pth', 'TrustOCT (Proposed)')
]

octid_rows = []
for config_item, weights_path, display_name in models_to_test_octid:
    if os.path.exists(weights_path) and len(df_octid) > 0:
        model_inst = build_model(config_item).to(device)
        model_inst.load_state_dict(torch.load(weights_path, map_location=device))
        model_inst.eval()
        
        is_ev = (config_item['model']['head'] == 'evidential')
        results = run_external_validation(
            model=model_inst,
            df_external=df_octid,
            batch_size=32,
            is_evidential=is_ev,
            device_name=str(device)
        )
        
        y_true = np.array(results['targets'])
        y_pred = np.array(results['predictions'])
        present_c = sorted(list(np.unique(y_true)))
        
        acc = accuracy_score(y_true, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=present_c, average='macro', zero_division=0)
        
        m = results['metrics']
        octid_rows.append({
            'Model Configuration': display_name,
            'OCTID Accuracy (%)': f"{acc*100:.2f}%",
            'Cohort Macro F1': f"{f1:.4f}",
            'ECE (Calibration)': f"{m['ece']:.4f}",
            'Brier Score': f"{m['brier_score']:.4f}"
        })

if len(octid_rows) > 0:
    table_5_df = pd.DataFrame(octid_rows)
    print('\n--- TABLE 5: CROSS-DATASET EXTERNAL GENERALIZATION (OCTID DATASET) ---')
    display(table_5_df)
    os.makedirs('results/tables', exist_ok=True)
    table_5_df.to_csv('results/tables/table_5_octid_generalization.csv', index=False)


## TABLE 6: Computational Complexity Analysis

In [ ]:
complexity_rows = []
dummy_input = torch.randn(1, 3, 224, 224).to(device)
models_for_complexity = [
    ({'model': {'backbone': 'resnet50', 'feature_module': 'identity', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/resnet50/weights_best.pth', 'ResNet-50 Baseline'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_resnet50/weights_best.pth', 'msf_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_resnet50/weights_best.pth', 'msf_cbam_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_mixstyle_resnet50/weights_best.pth', 'msf_cbam_mixstyle_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_edl_resnet50/weights_best.pth', 'msf_cbam_edl_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/trustoct_expB/weights_best.pth', 'TrustOCT (Proposed)')
]

for config_item, path, display_name in models_for_complexity:
    model_inst = build_model(config_item).to(device)
    if os.path.exists(path):
        model_inst.load_state_dict(torch.load(path, map_location=device))
        model_size_mb = os.path.getsize(path) / (1024 * 1024)
        size_str = f"{model_size_mb:.2f} MB"
    else:
        total_p = sum(p.numel() for p in model_inst.parameters())
        size_str = f"~{total_p * 4 / (1024*1024):.2f} MB"
        
    model_inst.eval()
    total_params = sum(p.numel() for p in model_inst.parameters())
    trainable_params = sum(p.numel() for p in model_inst.parameters() if p.requires_grad)
    
    with torch.no_grad():
        for _ in range(100):
            _ = model_inst(dummy_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            
        start_t = time.perf_counter()
        for _ in range(200):
            _ = model_inst(dummy_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end_t = time.perf_counter()
        
    avg_inf_ms = ((end_t - start_t) / 200) * 1000
    
    complexity_rows.append({
        'Model': display_name,
        'Total Params (M)': f"{total_params / 1e6:.2f}M",
        'Trainable Params (M)': f"{trainable_params / 1e6:.2f}M",
        'Size on Disk': size_str,
        'Inference Speed (ms)': f"{avg_inf_ms:.2f} ms"
    })

if len(complexity_rows) > 0:
    complexity_df = pd.DataFrame(complexity_rows)
    print('\n--- TABLE 6: COMPUTATIONAL COMPLEXITY ANALYSIS ---')
    display(complexity_df)
    os.makedirs('results/tables', exist_ok=True)
    complexity_df.to_csv('results/tables/table_6_computational_complexity.csv', index=False)


## Package & Download Results ZIP

In [ ]:
import shutil, os, torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Exact weight paths from Google Drive (TrustOCT_weights) ─────────────────
DRIVE_WEIGHTS_DIR = '/content/drive/MyDrive/TrustOCT_weights'

weight_map = [
    (os.path.join(DRIVE_WEIGHTS_DIR, 'resnet50.pth'),          'outputs/resnet50',          'Baseline ResNet-50'),
    (os.path.join(DRIVE_WEIGHTS_DIR, 'msf_resnet50.pth'),      'outputs/msf_resnet50',      'ResNet-50 + MSF'),
    (os.path.join(DRIVE_WEIGHTS_DIR, 'msf_cbam_resnet50.pth'), 'outputs/msf_cbam_resnet50', 'ResNet-50 + MSF + CBAM (Proposed)'),
]

print('--- SCANNING & COPYING DRIVE PRE-TRAINED WEIGHTS ---')
for src_p, folder, label in weight_map:
    os.makedirs(folder, exist_ok=True)
    dest_p = os.path.join(folder, 'weights_best.pth')
    if os.path.exists(src_p):
        shutil.copy(src_p, dest_p)
        size_mb = os.path.getsize(dest_p) / 1024 / 1024
        print(f'✅ {label:40} → {folder}  ({size_mb:.1f} MB)')
    else:
        # Fallback search in Google Drive
        alt_p = src_p.replace('TrustOCT_weights', 'TrustOCT_weights (1)')
        if os.path.exists(alt_p):
            shutil.copy(alt_p, dest_p)
            size_mb = os.path.getsize(dest_p) / 1024 / 1024
            print(f'✅ {label:40} → {folder}  ({size_mb:.1f} MB)')
        else:
            print(f'❌ Not found: {src_p}')
